# 레슨 02 — URL 파라미터와 페이지네이션

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/02/%5B%ED%95%99%EC%83%9D%EC%9A%A9%5D%20%EB%A0%88%EC%8A%A8%2002%20%E2%80%94%20URL%20%ED%8C%8C%EB%9D%BC%EB%AF%B8%ED%84%B0%EC%99%80%20%ED%8E%98%EC%9D%B4%EC%A7%80%EB%84%A4%EC%9D%B4%EC%85%98.ipynb)

이 레슨은 검색 URL을 문자열이 아니라 구조화된 데이터로 읽는 방법을 다룬다. 학생은 query string을 분해하고 다시 조립하며, 여러 장으로 나뉜 검색 결과를 안전하게 순회한다. 모든 예제는 수업용 합성 HTML fixture를 사용하므로 실제 웹사이트에 반복 요청을 보내지 않는다.

## 학습 목표

1. URL을 scheme, domain, path, query string으로 분해한다.
2. parse_qs, urlencode, urljoin으로 검색 조건과 상대 링크를 안전하게 다룬다.
3. 페이지 번호 규칙을 함수로 분리해 반복 수집 코드를 단순하게 만든다.
4. 페이지네이션 영역의 다음 링크와 현재 페이지 데이터를 구분한다.
5. 여러 페이지에서 모은 결과를 필터링, 집계, CSV 저장까지 연결한다.

---

## 1. URL은 문자열이 아니라 구조다

검색 페이지 URL은 길게 보이지만 실제로는 몇 개의 역할로 나뉜다. 예를 들어 https://example.com/library/search?q=python&category=all&page=2 에서 path는 /library/search이고, query string은 q, category, page 같은 검색 조건을 담는다.

자동화 코드를 만들 때 URL 전체를 문자열로 붙이면 실수하기 쉽다. 검색어에 공백이나 한글이 들어가면 직접 붙인 문자열은 깨질 수 있다. 그래서 파이썬에서는 먼저 URL을 분해하고, 조건은 딕셔너리로 관리한 다음 다시 안전하게 조립한다.


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def page_filename(page):
    return f'search_page_{page}.html'

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

sample_url = 'https://example.com/library/search?q=python&category=all&page=2'
parsed = urlparse(sample_url)
params = parse_qs(parsed.query)
print('path:', parsed.path)
print('query dict:', params)
print('page:', params['page'][0])


> **웹 자동화 안전 한스푼 — query string**
>
> - **뜻**: URL의 물음표 뒤에 붙는 검색 조건이다.
> - **왜 중요한가**: 검색어, 페이지 번호, 필터 조건이 여기 들어가므로 잘못 조립하면 다른 데이터를 가져온다.
> - **수업 기준**: query string은 직접 문자열로 이어 붙이지 않고 urlencode로 만든다.
> - **실수 예시**: q, page 값을 더하기 연산으로 붙이다가 공백과 숫자 처리를 놓친다.

---

## 2. 검색 조건을 딕셔너리로 관리하기

검색 자동화는 보통 같은 URL에 조건만 바꿔 여러 번 실행한다. 조건을 딕셔너리로 두면 검색어, 카테고리, 페이지 번호를 코드 안에서 명확하게 볼 수 있고, CSV에서 읽은 조건도 그대로 넣을 수 있다.

urlencode는 공백과 특수문자를 URL에 맞게 인코딩한다. 학생이 직접 퍼센트 인코딩을 외울 필요가 없다.


In [ ]:
base_url = 'https://example.com/library/search'
query = {'q': 'python automation', 'category': 'course', 'page': 1}
url = base_url + '?' + urlencode(query)
print(url)

query['page'] = 2
print(base_url + '?' + urlencode(query))


---

## 3. fixture HTML 읽기

이번 레슨의 검색 결과는 search_page_1.html, search_page_2.html, search_page_3.html 세 파일로 준비되어 있다. 각 파일에는 article.result-card 9개가 들어 있고, 각 카드에는 제목, 상세 링크, 카테고리, 날짜, 조회수가 있다.

실제 사이트를 요청하지 않고 파일 fixture를 쓰는 이유는 수업 중 같은 결과를 안정적으로 재현하기 위해서다. 학생이 반복 실행해도 외부 서버에 부하가 가지 않고, 선생님은 selector 오류와 코드 오류를 구분해서 지도할 수 있다.


In [ ]:
html_text = load_text('search_page_1.html')
soup = BeautifulSoup(html_text, 'html.parser')
heading = soup.select_one('h1').text.strip()
cards = soup.select('article.result-card')
first = cards[0]
print(heading)
print('card count:', len(cards))
print(first.select_one('.title a').text.strip())
print(first['data-category'], first['data-page'])


> **웹 자동화 안전 한스푼 — fixture**
>
> - **뜻**: 수업이나 테스트를 위해 고정해 둔 샘플 데이터다.
> - **왜 중요한가**: 외부 사이트 상태가 바뀌어도 수업 결과가 흔들리지 않는다.
> - **수업 기준**: 1~5강은 실제 사이트 대신 합성 fixture로 selector와 반복 구조를 연습한다.
> - **실수 예시**: fixture에서 충분히 연습하지 않고 실제 사이트를 빠르게 반복 요청한다.

---

## 4. 카드 하나를 레코드로 바꾸기

HTML 카드 하나를 그대로 저장하면 나중에 정렬하거나 필터링하기 어렵다. 자동화 결과는 사람이 읽는 화면에서 파이썬이 다루기 쉬운 딕셔너리로 바꾸는 과정이 필요하다.

여기서는 제목, 카테고리, 페이지 번호, 날짜, 조회수, 상세 URL을 하나의 딕셔너리로 만든다. 조회수는 조회 1,023 같은 문자열이므로 숫자만 남겨 정수로 바꾼다. 링크는 /library/... 형태의 상대 경로이므로 urljoin으로 절대 URL을 만든다.


In [ ]:
def parse_result_card(card, base='https://example.com'):
    link = card.select_one('.title a')
    return {
        'title': link.text.strip(),
        'category': card['data-category'],
        'page': int(card['data-page']),
        'rank': int(card['data-rank']),
        'date': card.select_one('time')['datetime'],
        'views': clean_int(card.select_one('.views').text),
        'url': urljoin(base, link['href']),
        'detail_url': urljoin(base, card.select_one('a.detail')['href']),
    }

record = parse_result_card(first)
print(record)


> **웹 자동화 안전 한스푼 — 상대 URL**
>
> - **뜻**: /library/web-resource-1처럼 도메인 없이 경로만 있는 링크다.
> - **왜 중요한가**: 그대로 저장하면 어느 사이트의 링크인지 알 수 없다.
> - **수업 기준**: 저장 전 urljoin(base_url, href)로 절대 URL을 만든다.
> - **실수 예시**: 상대 경로만 CSV에 저장해 다음 자동화 단계에서 링크를 열 수 없다.

---

## 5. 한 페이지를 리스트로 정리하기

자동화에서 반복 단위가 정해지면 다음 단계는 리스트를 만드는 것이다. 한 페이지 안의 모든 article.result-card를 parse_result_card 함수로 바꾸면 결과는 딕셔너리 리스트가 된다.

이때 len(records)를 먼저 확인하는 습관이 중요하다. 첫 번째 값만 출력하면 selector가 일부만 맞아도 지나칠 수 있지만, 개수를 확인하면 fixture 구조와 코드가 맞는지 빠르게 판단할 수 있다.


In [ ]:
page1_records = [parse_result_card(card) for card in cards]
print('records:', len(page1_records))
print(page1_records[0]['title'], page1_records[0]['views'])
print(page1_records[-1]['title'], page1_records[-1]['views'])


---

## 6. 페이지 번호 규칙을 함수로 분리하기

이번 fixture의 파일명은 search_page_1.html, search_page_2.html, search_page_3.html이다. 페이지 번호가 들어가는 자리를 함수로 분리하면 반복문에서 파일명을 직접 조립하지 않아도 된다.

함수로 분리하는 이유는 단순히 코드가 짧아지기 때문만이 아니다. 실제 사이트에서 페이지 규칙이 바뀌거나 파일명 규칙이 바뀌면 함수 한 곳만 수정하면 된다.


In [ ]:
def page_filename(page):
    return f'search_page_{page}.html'

for page in range(1, 4):
    print(page, page_filename(page))


---

## 7. 여러 페이지 순회하기

여러 페이지를 순회할 때는 바깥 반복문이 페이지를 바꾸고, 안쪽 반복문이 카드들을 처리한다. 이 구조를 분명히 이해해야 나중에 페이지네이션이 많은 사이트에서도 무한 반복을 피할 수 있다.

수업에서는 1~3페이지로 고정하지만, 실제 운영 코드에서는 다음 링크 존재 여부, 결과 개수, 최대 페이지 수 같은 중단 기준을 반드시 둔다.


In [ ]:
all_records = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    page_cards = page_soup.select('article.result-card')
    print('page', page, 'cards', len(page_cards))
    for card in page_cards:
        all_records.append(parse_result_card(card))

print('total:', len(all_records))
print(all_records[0]['title'], '->', all_records[-1]['title'])


> **웹 자동화 안전 한스푼 — 페이지네이션 중단 기준**
>
> - **뜻**: 반복 수집을 언제 멈출지 정하는 규칙이다.
> - **왜 중요한가**: 중단 기준이 없으면 같은 요청을 무한히 반복할 수 있다.
> - **수업 기준**: fixture는 1~3페이지로 제한하고, 실제 사이트는 최대 페이지 수나 다음 링크 존재 여부를 확인한다.
> - **실수 예시**: 무한 반복으로 계속 다음 페이지를 요청하고 빈 결과를 확인하지 않는다.

---

## 8. 다음 페이지 링크 읽기

페이지 번호를 직접 증가시키는 방식과 별개로, HTML 안의 페이지네이션 영역을 읽을 수도 있다. nav.pagination a.next는 다음 페이지 링크를 제공한다. 이 링크에서 다시 query string을 읽으면 다음 page 값을 확인할 수 있다.


In [ ]:
next_link = soup.select_one('nav.pagination a.next')
next_href = next_link['href']
next_params = parse_qs(urlparse(next_href).query)
print(next_href)
print('next page:', next_params['page'][0])


---

## 9. 필터링과 집계

수집한 데이터는 저장하기 전에 작은 검증과 요약을 거친다. 예를 들어 조회수 1000 이상인 자료만 골라보거나, 카테고리별 개수를 세면 selector가 의도대로 작동했는지 확인할 수 있다.


In [ ]:
popular = [row for row in all_records if row['views'] >= 1000]
print('popular:', len(popular))
print([row['title'] for row in popular[:5]])

category_counts = {}
for row in all_records:
    category = row['category']
    category_counts[category] = category_counts.get(category, 0) + 1
print(category_counts)


---

## 10. 검색 계획 CSV 읽기

search_targets.csv는 검색어, 카테고리, 최소 조회수, 최대 페이지 수를 담은 계획 파일이다. 실제 업무에서는 검색 조건을 코드 안에 박아두기보다 CSV나 설정 파일로 분리하는 편이 관리하기 쉽다.


In [ ]:
target_rows = list(csv.DictReader(load_text('search_targets.csv').splitlines()))
for row in target_rows:
    print(row['query'], row['category'], row['min_views'], row['max_pages'])


---

## 11. CSV로 저장하기

마지막으로 여러 페이지에서 모은 데이터를 CSV로 저장한다. 저장 전에는 필드 이름을 먼저 정한다. 필드 이름이 일정해야 나중에 엑셀, 구글시트, 데이터 분석 코드에서 같은 구조로 읽을 수 있다.


In [ ]:
fieldnames = ['title', 'category', 'page', 'date', 'views', 'url']
with open('lesson02_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows({key: row[key] for key in fieldnames} for row in all_records)
print('saved:', 'lesson02_results.csv', len(all_records))


---

## 데이터 출처와 안전 규칙

이 레슨의 파일은 모두 수업용 합성 데이터다. 실제 사이트의 개인정보, 로그인 정보, 유료 콘텐츠를 포함하지 않는다. 실제 사이트로 확장할 때는 robots.txt, 이용 약관, 요청 간격, 수집 목적을 먼저 확인한다. 수업 중에는 fixture를 반복 실행하며 구조를 익히고, 외부 사이트를 빠르게 반복 요청하지 않는다.

---

## 12. 디버깅 순서

페이지네이션 자동화가 실패했을 때는 selector부터 고치지 않는다. 먼저 조합된 URL이 의도한 조건을 담고 있는지 확인한다. 그다음 HTML을 읽었는지, 반복 단위 개수가 예상과 맞는지, 카드 안에서 필요한 값이 빠지지 않았는지 차례로 확인한다.

수업 중에는 아래 순서를 그대로 말하게 한다.

1. 현재 page 번호와 파일명 또는 URL을 출력한다.
2. HTML 제목 h1을 출력해 올바른 페이지를 읽었는지 확인한다.
3. article.result-card 개수를 출력한다.
4. 첫 카드의 title, href, data-category를 출력한다.
5. 조회수를 정수로 바꾼 뒤 타입을 확인한다.
6. 저장 전 rows 길이와 첫 행의 key 목록을 확인한다.

이 순서를 지키면 학생이 selector를 무작정 바꾸는 시간을 줄일 수 있다. 특히 페이지 반복에서는 첫 페이지 결과가 세 번 들어가는 실수가 자주 나오므로 page_soup을 반복문 안에서 새로 만드는지 확인한다.

## 13. 실제 사이트로 확장하기 전 점검

이번 레슨은 fixture만 사용하지만, 실제 사이트로 확장할 때는 코드보다 운영 기준을 먼저 정해야 한다. 검색 결과가 공개 페이지인지, robots 정책에서 차단하지 않는지, 요청 간격을 둘 수 있는지, 수집한 값을 저장해도 되는지 확인한다. 학생에게는 “코드가 된다”와 “운영해도 된다”가 다르다는 점을 반복해서 설명한다.

robots_sample.txt는 실제 법적 판단을 대신하지 않는다. 수업에서는 허용/차단/지연 요청의 개념을 익히는 샘플로만 사용한다. 실제 서비스에서는 사이트 약관, 관리자 허가, 개인정보 여부를 함께 검토해야 한다.

## 14. 수업 중 확인 질문

- query string에서 q, category, page는 각각 어떤 역할인가?
- 검색어에 공백이 들어갈 때 urlencode가 필요한 이유는 무엇인가?
- article.result-card 대신 a 태그를 반복 단위로 잡으면 어떤 문제가 생기는가?
- 상대 URL을 그대로 저장하면 다음 자동화 단계에서 어떤 정보가 부족한가?
- range(1, 4)가 1, 2, 3을 만든다는 사실을 어디에서 확인할 수 있는가?
- 조회수 문자열을 정수로 바꾸지 않으면 필터링 결과가 왜 틀릴 수 있는가?
- CSV 저장 전에 rows 길이와 fieldnames를 확인하는 이유는 무엇인가?

## 15. 이번 레슨의 완성 기준

학생이 완성해야 하는 것은 단순히 CSV 파일 하나가 아니다. URL 조건을 구조화하고, 페이지 반복 범위를 통제하고, 카드 데이터를 같은 딕셔너리 구조로 맞춘 뒤, 저장 전 간단한 검증을 하는 흐름이다. 이 네 가지가 연결되어야 다음 레슨의 테이블/리스트 데이터 정리로 자연스럽게 넘어갈 수 있다.

---

## 16. 예제 데이터를 읽는 관찰 포인트

search_page 파일들은 일부러 같은 구조를 유지하면서 값만 다르게 만들었다. 학생은 먼저 “무엇이 반복되고 무엇이 달라지는지”를 말해야 한다. 반복되는 것은 article.result-card, .title a, .views, time 태그이고, 달라지는 것은 data-category, data-page, data-rank, 제목, 조회수다.

이 관찰을 먼저 하지 않으면 selector를 외워서 쓰게 된다. 수업에서는 코드를 치기 전에 HTML 일부를 읽고, 반복 단위와 필요한 필드를 표로 정리하게 한다. 이 과정이 있어야 3강에서 table, list, card 구조가 바뀌어도 같은 방식으로 접근할 수 있다.

## 17. 저장 파일을 검토하는 기준

CSV 저장 후에는 파일이 만들어졌다는 사실만 보지 않는다. 첫 줄에 header가 있는지, title/category/views 같은 필드 이름이 일관적인지, 행 수가 예상한 카드 수와 맞는지 확인한다. 이 레슨에서는 3페이지와 페이지당 9개 카드이므로 전체 rows는 27개가 되어야 한다.

필터링 결과는 기준에 따라 달라질 수 있다. 그래서 최소 조회수 기준을 코드 주석이나 결과 요약에 남겨야 한다. 운영자가 다음에 같은 자동화를 실행할 때 기준을 모르면 결과 차이를 오류로 오해할 수 있다.


# 레슨 02 — 실습 문제

URL 파라미터와 페이지네이션 레슨의 학생용 문제 노트북이다. 강의 노트북을 먼저 실행한 뒤 빈칸을 직접 채운다. 문제는 URL 구조 이해에서 시작해 여러 페이지 수집과 CSV 저장까지 이어진다.

## 통과 기준

- 총 15문제 중 12문제 이상 정상 출력이면 통과.
- 문제 1~5는 URL과 첫 페이지 구조 확인, 6~10은 링크 정리와 페이지 반복, 11~15는 필터링, 집계, 저장이다.
- 정답값과 완성 코드는 적지 않는다. 출력 형태와 fixture 구조를 보고 직접 판단한다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))

def page_filename(page):
    return f'search_page_{page}.html'

print('colab:', IS_COLAB)
print('data base:', DATA_BASE)


---

## 문제 1 — 검색 URL 분해하기

예시 URL에서 path와 query string 값을 분리한다. URL 전체를 문자열로만 보지 않고 검색 조건을 딕셔너리로 읽는 연습이다.

**기대 결과 형태**: path 한 줄과 q/category/page 값 한 줄이 출력된다.

**빈칸 힌트**: urlparse는 URL 구조를 나누고, parse_qs는 query string을 딕셔너리 형태로 바꾼다.


In [ ]:
sample_url = 'https://example.com/library/search?q=python&category=all&page=2'
parsed = ____(sample_url)
params = ____(parsed.query)
print(parsed.____)
print(params['____'][0], params['____'][0], params['____'][0])


---

## 문제 2 — 쿼리 문자열 만들기

검색 조건을 딕셔너리로 만든 뒤 URL에 붙인다. 공백이 들어간 검색어가 깨지지 않도록 안전한 조립 방식을 사용한다.

**기대 결과 형태**: 검색어, 카테고리, 페이지 번호가 들어간 URL 문자열이 출력된다.

**빈칸 힌트**: 조건은 딕셔너리로 만들고, query string 변환에는 urlencode를 쓴다.


In [ ]:
base_url = 'https://example.com/library/search'
query = {'q': ____, 'category': ____, 'page': ____}
url = base_url + '?' + ____(query)
print(url)


---

## 문제 3 — 첫 페이지 HTML 읽기

search_page_1.html fixture를 읽고 BeautifulSoup 객체로 변환한다. 이후 문제에서 같은 soup 변수를 재사용한다.

**기대 결과 형태**: 첫 페이지의 h1 텍스트가 출력된다.

**빈칸 힌트**: 파일은 load_text로 읽고, HTML parser는 BeautifulSoup(..., html.parser) 형태로 만든다.


In [ ]:
html_text = ____(____)
soup = ____(html_text, 'html.parser')
print(soup.____('____').text.strip())


---

## 문제 4 — 결과 카드 개수 세기

검색 결과의 반복 단위인 article.result-card를 모두 선택하고 개수를 확인한다.

**기대 결과 형태**: 카드 개수가 숫자로 출력된다.

**빈칸 힌트**: 반복 단위는 article 태그와 result-card class를 함께 사용한다.


In [ ]:
cards = soup.____('____')
print('cards:', ____)


---

## 문제 5 — 첫 카드 제목과 href 읽기

첫 번째 카드에서 제목 텍스트와 링크 경로를 읽는다. 카드 내부 selector를 쓰는 연습이다.

**기대 결과 형태**: 제목 한 줄과 상대 링크 한 줄이 출력된다.

**빈칸 힌트**: 제목 링크는 .title a 위치에 있다. href는 태그 속성으로 읽는다.


In [ ]:
first = cards[____]
title = first.____('____').text.strip()
href = first.____('____')['____']
print(title)
print(href)


---

## 문제 6 — 상대 URL을 절대 URL로 바꾸기

카드에서 읽은 상대 링크를 도메인이 포함된 절대 URL로 변환한다.

**기대 결과 형태**: https://example.com으로 시작하는 URL이 출력된다.

**빈칸 힌트**: 기준 도메인과 상대 경로를 합칠 때는 urljoin을 사용한다.


In [ ]:
absolute = ____('https://example.com', ____)
print(absolute)


---

## 문제 7 — 카드 하나를 딕셔너리로 만들기

첫 카드에서 제목, 카테고리, 날짜, 조회수, 절대 URL을 뽑아 하나의 딕셔너리로 정리한다.

**기대 결과 형태**: title, category, date, views, url 키를 가진 딕셔너리가 출력된다.

**빈칸 힌트**: 카테고리는 data-category, 날짜는 time의 datetime, 조회수는 clean_int로 정리한다.


In [ ]:
item = {
    'title': first.select_one('____').text.strip(),
    'category': first['____'],
    'date': first.select_one('____')['____'],
    'views': ____(first.select_one('____').text),
    'url': urljoin('https://example.com', first.select_one('____')['____']),
}
print(item)


---

## 문제 8 — 한 페이지 결과 리스트 만들기

첫 페이지의 모든 카드를 순회해 딕셔너리 리스트로 만든다.

**기대 결과 형태**: 첫 번째 레코드와 전체 레코드 수가 출력된다.

**빈칸 힌트**: 반복문 안에서 같은 필드를 만들어 results에 추가한다.


In [ ]:
results = []
for card in cards:
    results.append({
        'title': card.select_one('____').text.strip(),
        'category': card['____'],
        'page': int(card['____']),
        'views': ____(card.select_one('____').text),
    })
print(results[0])
print(len(results))


---

## 문제 9 — 페이지 번호로 파일명 만들기

페이지 번호를 받아 fixture 파일명을 반환하는 함수를 만든다.

**기대 결과 형태**: 1, 2, 3페이지 파일명이 차례대로 출력된다.

**빈칸 힌트**: 파일명은 search_page_번호.html 규칙이다.


In [ ]:
def page_filename(page):
    return f'_____{page}.____'

for page in [1, 2, 3]:
    print(____(page))


---

## 문제 10 — 3페이지 전체 순회하기

1~3페이지 fixture를 순회하면서 모든 카드 제목을 하나의 리스트로 모은다.

**기대 결과 형태**: 전체 제목 개수와 마지막 제목이 출력된다.

**빈칸 힌트**: range(1, 4)는 1, 2, 3을 만든다. 각 페이지마다 page_filename(page)로 파일을 읽는다.


In [ ]:
all_results = []
for page in range(____, ____):
    page_soup = BeautifulSoup(load_text(____(page)), 'html.parser')
    for card in page_soup.select('____'):
        all_results.append(card.select_one('____').text.strip())
print(len(all_results))
print(all_results[-1])


---

## 문제 11 — 다음 페이지 링크 찾기

pagination 영역의 다음 링크를 읽고 그 안의 page 값을 확인한다.

**기대 결과 형태**: 다음 링크 경로와 다음 페이지 번호가 출력된다.

**빈칸 힌트**: 다음 링크는 a.next이고, page 값은 href의 query string 안에 있다.


In [ ]:
next_href = soup.select_one('____')['____']
next_query = ____(____(next_href).query)
print(next_href)
print(next_query['____'][0])


---

## 문제 12 — 조회수 1000 이상 필터링

모든 페이지를 순회하면서 조회수가 1000 이상인 자료 제목만 모은다.

**기대 결과 형태**: 조건을 만족하는 제목 리스트가 출력된다.

**빈칸 힌트**: 조회수는 .views 텍스트를 clean_int로 바꾼 뒤 비교한다.


In [ ]:
rich = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        views = ____(card.select_one('____').text)
        if views >= ____:
            rich.append(card.select_one('____').text.strip())
print(rich)


---

## 문제 13 — 카테고리별 개수 세기

모든 페이지의 카드에서 data-category를 읽어 카테고리별 개수를 집계한다.

**기대 결과 형태**: 카테고리를 키로 가진 딕셔너리가 출력된다.

**빈칸 힌트**: 딕셔너리에 없는 키는 get(key, 0)으로 시작값을 만든다.


In [ ]:
counts = {}
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        key = card['____']
        counts[key] = counts.get(key, ____) + ____
print(counts)


---

## 문제 14 — targets CSV 읽기

검색 계획 파일인 search_targets.csv를 읽어 검색어와 최대 페이지 수를 확인한다.

**기대 결과 형태**: 각 행의 query와 max_pages 값이 출력된다.

**빈칸 힌트**: csv.DictReader는 첫 줄 헤더를 딕셔너리 키로 사용한다.


In [ ]:
target_rows = list(csv.DictReader(load_text('____').splitlines()))
for row in target_rows:
    print(row['____'], row['____'])


---

## 문제 15 — 검색 결과 CSV 저장하기

모든 페이지의 제목, 카테고리, 조회수를 모아 CSV 파일로 저장한다.

**기대 결과 형태**: 저장 파일명과 저장 행 수가 출력된다.

**빈칸 힌트**: DictWriter는 fieldnames, writeheader, writerows 순서로 사용한다.


In [ ]:
rows = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        rows.append({
            'title': card.select_one('____').text.strip(),
            'category': card['____'],
            'views': ____(card.select_one('____').text),
        })
with open('lesson02_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['title', 'category', 'views'])
    writer.____()
    writer.____(rows)
print('saved:', 'lesson02_results.csv', len(rows))


---

## 풀이 전략

문제 1~5는 첫 페이지를 정확히 읽는 단계다. 여기서 soup, cards, first, href 변수를 제대로 만들어야 뒤 문제가 자연스럽게 이어진다. 변수가 비어 있으면 다음 문제를 억지로 풀지 말고, h1 출력과 카드 개수부터 다시 확인한다.

문제 6~10은 카드 하나를 저장 가능한 데이터로 바꾸고, 같은 작업을 여러 페이지로 확장하는 단계다. 상대 URL, 숫자 변환, page_filename 함수는 이후 문제에서 계속 쓰인다. 함수 이름과 변수 이름을 임의로 바꾸면 뒤 셀을 실행할 때 NameError가 날 수 있다.

문제 11~15는 운영자가 쓰는 형태로 정리하는 단계다. 다음 페이지 링크를 읽고, 조회수 기준으로 필터링하고, 카테고리별로 집계하고, CSV로 저장한다. 출력값만 맞히는 것보다 어떤 기준으로 걸렀는지 설명하는 것이 더 중요하다.

## 제출 전 자체 점검

- 검색 URL을 직접 split하지 않고 urlparse와 parse_qs로 분리했는가?
- 검색 조건 URL을 직접 이어 붙이지 않고 urlencode를 사용했는가?
- 모든 HTML 파일은 load_text로 읽었는가?
- 반복 단위는 article.result-card로 잡았는가?
- 조회수는 clean_int로 정수 변환했는가?
- 상대 링크는 urljoin으로 절대 URL로 바꿨는가?
- 1, 2, 3페이지를 모두 순회했는가?
- CSV 저장 전에 rows 길이와 fieldnames가 맞는지 확인했는가?

## 막혔을 때 확인할 값

첫 번째로 page_filename(1)의 결과를 확인한다. 두 번째로 search_page_1.html의 h1을 출력한다. 세 번째로 len(cards)를 출력한다. 네 번째로 cards[0]에서 title과 href가 나오는지 확인한다. 이 네 값이 맞으면 대부분의 문제는 selector나 타입 변환 실수다.

---

## 문제별 변수 연결표

| 구간 | 핵심 변수 | 뒤에서 쓰이는 곳 |
|---|---|---|
| 문제 1~2 | parsed, params, url | URL 구조 설명과 검색 조건 조립 |
| 문제 3~5 | soup, cards, first, href | 첫 페이지 selector 확인 |
| 문제 6~8 | absolute, item, results | 저장 가능한 레코드 구조 만들기 |
| 문제 9~10 | page_filename, all_results | 여러 페이지 반복 |
| 문제 11~15 | next_query, rich, counts, target_rows, rows | 운영 요약과 CSV 저장 |

변수는 정답을 맞히기 위한 임시 이름이 아니라 다음 셀과 연결되는 약속이다. 이름을 바꿔도 되지만, 바꾼 뒤에는 뒤 셀에서도 같은 이름으로 맞춰야 한다. 수업 중에는 NameError가 나면 바로 정답을 보지 말고 어느 문제에서 변수가 만들어졌는지 먼저 거슬러 올라간다.

## 힌트 사용 기준

힌트는 완성 코드를 알려주는 용도가 아니다. 어떤 HTML 구조를 봐야 하는지, 어떤 표준 함수를 떠올려야 하는지 알려주는 방향으로만 사용한다. 빈칸을 채울 때는 바로 실행하지 말고, 그 빈칸이 함수 이름인지 selector인지 속성 이름인지 먼저 구분한다.

---

## 최소 통과 후 심화 방향

12문제 이상 통과한 학생은 CSV 저장 후 category가 python인 행만 다시 골라본다. 그다음 views 기준으로 내림차순 정렬해 상위 3개 제목을 출력한다. 이 심화는 새 문법을 요구하지 않고, 이미 만든 rows 구조를 재사용하는 연습이다.


# 레슨 02 — 최종 미션

여러 페이지로 나뉜 자료실 검색 결과를 읽어 운영자가 확인할 수 있는 CSV를 만든다. 이번 미션은 실제 사이트 요청 없이 search_page_1.html부터 search_page_3.html까지의 fixture만 사용한다.

## 시나리오

학원 운영자가 수업 자료실에서 자동화 관련 자료를 모아보고 싶어 한다. 검색 결과는 페이지별 HTML로 저장되어 있고, 검색 계획은 search_targets.csv에 있다. 학생은 URL 조건을 읽고, 페이지별 검색 결과를 파싱하고, 조회수 기준으로 중요한 자료를 추려 CSV로 저장해야 한다.

## 필수 요구사항

1. search_targets.csv를 읽어 검색어, 카테고리, 최소 조회수, 최대 페이지 수를 확인한다.
2. search_page_1.html부터 search_page_3.html까지 순회한다.
3. 각 카드에서 제목, 카테고리, 날짜, 조회수, 상세 URL을 추출한다.
4. 상대 링크는 https://example.com 기준의 절대 URL로 변환한다.
5. 조회수가 기준 이상인 자료만 별도 리스트로 만든다.
6. 전체 결과와 필터링 결과를 CSV로 저장한다.

## 보너스 요구사항

- 카테고리별 자료 개수와 평균 조회수를 출력한다.
- pagination의 a.next 링크를 읽어 다음 페이지 번호를 확인하는 함수를 만든다.

## 제출 산출물

- 실행 가능한 노트북
- 결과 CSV 1개 이상
- 자동화 결과 요약 3문장
- 실제 사이트로 확장할 때 지킬 안전 규칙 2개

## 스타터 코드


In [ ]:
# 여러 페이지를 순회해 summary_rows를 만든다.
summary_rows = []
# TODO: search_targets.csv를 읽고, HTML fixture를 파싱하고, CSV로 저장한다.


## 자동화 결과 요약

- 수집 대상:
- 핵심 결과:
- 다음 실행 때 조심할 점:

---

## 수행 순서 제안

1. search_targets.csv를 먼저 출력해 검색 계획의 컬럼을 확인한다.
2. parse_result_card 함수를 만들어 카드 하나를 같은 구조의 딕셔너리로 바꾼다.
3. page_filename 함수로 1~3페이지 파일명을 만든다.
4. 모든 페이지를 순회해 all_rows를 만든다.
5. 조회수 기준으로 filtered_rows를 만든다.
6. 두 결과를 CSV로 저장한다.
7. 마지막에 수집 대상, 핵심 결과, 다음 실행 때 조심할 점을 3문장으로 정리한다.

## 평가 루브릭

| 항목 | 통과 기준 |
|---|---|
| URL 구조 이해 | query string을 표준 함수로 분해하거나 조립한다. |
| 반복 단위 | article.result-card를 기준으로 모든 페이지를 순회한다. |
| 데이터 정리 | 제목, 카테고리, 날짜, 조회수, URL이 같은 key 구조로 저장된다. |
| 안전 기준 | 실제 사이트에 요청하지 않고 fixture만 사용한다. |
| CSV 저장 | 헤더가 있는 CSV를 만들고 저장 행 수를 출력한다. |
| 설명 | 결과 요약과 다음 실행 시 주의할 점을 작성한다. |

## 감점 기준

- 실제 사이트 URL을 반복 요청한다.
- 첫 페이지 데이터만 저장한다.
- 조회수를 문자열로 둔 채 필터링한다.
- 상대 링크를 그대로 저장한다.
- CSV 헤더 없이 문자열을 직접 이어 붙여 저장한다.
- 결과 요약 없이 코드 실행 결과만 제출한다.

## 제출 전 검증

최종 제출 전에는 all_rows의 길이, filtered_rows의 길이, CSV 파일명, 첫 행의 key 목록을 출력한다. 이 네 가지가 확인되면 선생님이 파일을 열기 전에 구조 오류를 빠르게 찾을 수 있다.

---

## 결과 요약 예시 형식

아래 형식을 참고하되 숫자와 문장은 본인이 실행한 결과에 맞게 적는다.

- 수집 대상: 수업용 자료실 검색 fixture 3페이지를 순회했다.
- 핵심 결과: 전체 결과와 조회수 기준 필터링 결과를 CSV로 저장했다.
- 다음 실행 때 조심할 점: 실제 사이트에서는 요청 간격, robots 정책, 개인정보 포함 여부를 먼저 확인해야 한다.

## 선생님 확인 항목

- 학생이 실제 사이트를 호출하지 않았는가?
- 결과 CSV에 header가 있는가?
- 상대 URL이 절대 URL로 변환되었는가?
- 조회수 비교 전에 정수 변환이 되었는가?
- 페이지 1~3을 모두 순회했는가?
- 결과 요약이 코드 실행 결과와 일치하는가?

최종 미션은 코드만 제출하는 과제가 아니다. 운영자가 다시 실행할 수 있는 자동화 절차를 만들고, 그 결과를 짧은 문장으로 설명하는 과제다.

---

## 확장 아이디어

시간이 남으면 category별 CSV를 따로 저장해본다. 예를 들어 python 자료만 lesson02_python.csv로 저장하고, web 자료만 lesson02_web.csv로 저장한다. 이 확장은 실제 운영에서 담당자별 자료 목록을 나누어 전달하는 상황과 연결된다.
